# Step 1 - synthetic data generation

Hill-type ODEs in, 20 pairs of ground-truth networks and their trajectories out.

**Re-running this overwrites `../synthetic_data` and will not reproduce the
committed trajectories** - the generator was edited after they were written, so
a re-run would not match figure 2. The network structures do reproduce.

In [ ]:
import os
import sys

# paths below are relative to ../synthetic_data
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "synthetic_data"))

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Callable, Dict, Optional, Literal, Tuple
from scipy.integrate import solve_ivp
from sklearn.preprocessing import MinMaxScaler
import math
import random
from pathlib import Path

os.chdir(DATA_DIR)

## The ODE model

In [ ]:
def hill_function(xj, lam_ij, R_ij, n_ij):
    return lam_ij + (1 - lam_ij) / (1 + (xj / R_ij) ** (n_ij))

#force gene4 to take these values
def forced_function(t, t_start, t_end, x_start, x_end):
    if t < t_start:
        return x_start
    if t > t_end:
        return x_end
        
    k = 0.5 * (1.0 - math.cos(1.5 * math.pi * (t - t_start) / (t_end - t_start)))
    return x_start + (x_end - x_start) * k

#t_val is time point, gene_vals are values for all genes at that time point, forced is index of gene that is forced
#forced_vals is the list of values we choose
def hill_rhs(t_val, gene_vals, params, forced, forced_params):
    g = params["g"]
    k = params["k"]
    l= params["l"]
    R = params["R"]
    n = params["n"]

    gene_count = len(gene_vals)
    
    dx = np.zeros(gene_count, dtype=float)
    
    for i in range(gene_count):
        if i == forced:
            dx[i] = 0.0
        else:
            prod = 1.0
            for j in range(gene_count):
                if j == forced:
                    x_forced = forced_function(t_val, forced_params[0], forced_params[1], forced_params[2], forced_params[3])
                    prod *= hill_function(x_forced, l[i, j], R[i, j], n[i, j])
                else: 
                    prod *= hill_function(gene_vals[j], l[i, j], R[i, j], n[i, j])
            dx[i] = g[i] * prod - k[i] * gene_vals[i]
    return dx

def simulate(initial_x, t_start, t_end, params, forced, forced_params, steps = 1000):
    initial_x = np.asarray(initial_x)
    timepoints = np.linspace(t_start, t_end, steps)
    gene_count = len(initial_x)
    gene_names = ["gene_" + str(i+1) for i in range(gene_count)]

    #solving ODE 
    sol = solve_ivp(hill_rhs, (t_start, t_end), initial_x, t_eval = timepoints, 
                      method = "LSODA", args = (params, forced, forced_params))

    raw = sol.y
    t0, t1, x_start, x_end = forced_params
    u = np.array([forced_function(t, t0, t1, x_start, x_end) for t in timepoints])
    u = np.asarray(u)
    raw[forced, :] = u

    raw_df = pd.DataFrame(raw, index=gene_names, columns=timepoints)
    log_df = pd.DataFrame(np.log2(raw + 1.0), index=gene_names, columns=timepoints)
    return raw_df, log_df

## Steady state and threshold clipping

In [ ]:
def find_initial_x(params, forced, forced_value, relax_time=1000.0, steps = 1000):
    gene_count = len(params["g"])
    x0 = np.array([random.uniform(0.0, 50.0) for i in range(gene_count)])
    x0[forced] = forced_value

    forced_params = (0.0, 1.0, forced_value, forced_value)
    raw_df, log_df = simulate(x0, 0.0, relax_time, params, forced, forced_params, steps)
    
    x_initial = raw_df.values[:, -1].copy()
    x_initial[forced] = forced_value
    return x_initial, raw_df

def clip_R(params, df):
    R = params["R"]
    df = df.values.astype(float)           
    mn = df.min(axis=1)   
    mx = df.max(axis=1) 
    for i in range(len(R)):
        for j in range(len(R[0, ])):
            if R[i, j] < mn[j] or R[i, j] > mx[j]:
                #go from 0.1 to 0.5
                #lo = 0.25*(3*mn[j]+mx[j])
                #hi = 0.25*(mn[j]+3*mx[j])
                lo = 0.9*mn[j] + 0.1*mx[j]
                hi = 0.5*mn[j] + 0.5*mx[j]
                R[i, j] = random.uniform(lo, hi)
    params["R"] = R
    return params

def update_params(params, forced, forced_value,
                 relax_time=200.0, steps=1000):
    #clip R from steady state
    x_init, df = find_initial_x(params, forced, forced_value,
                            relax_time=relax_time, steps=steps)
    params = clip_R(params, df)    

    #find new x_init from new R's
    x_init, df = find_initial_x(params, forced, forced_value, relax_time, steps)
    params = clip_R(params, df)    

    #clip R again from dynamic simulation
    forced_params = (0.0, relax_time, forced_value, 6*forced_value)
    raw_df, log_df = simulate(x_init, 0.0, relax_time, params, forced, forced_params, steps)
    params = clip_R(params, raw_df)

    #find final good inital x
    x_init, df = find_initial_x(params, forced, forced_value, relax_time, steps)      
    raw_df, log_df = simulate(x_init, 0.0, relax_time, params, forced, forced_params, steps)
    return x_init, params, raw_df

## Sampling a pair of similar networks

In [ ]:
def log_rand(a, b):
    return 10 ** random.uniform(math.log10(a), math.log10(b))


def get_lam(percent_inhibition, min_inhibition = 0.05, max_inhibition = 0.5, min_activation = 2, max_activation = 20):
    if random.random() < percent_inhibition:
        return(log_rand(min_inhibition, max_inhibition))
    else:
        return(log_rand(min_activation, max_activation))


#given number of edges, forced index, and minimum #regulators + #regulations for each gene
def build_edges(n, edges, forced = 3, min_in = 1, min_out = 1):
    M = np.ones((n, n), dtype=float)
    chosen = set()
    row_count = [0 for i in range(n)]
    col_count = [0 for j in range(n)]
    #satisfy min regulators for gene
    for i in range(n):
        if i != forced:
            while row_count[i] < min_in:
                j = random.randrange(n)
                if (i, j) not in chosen and i != j:
                    chosen.add((i,j))
                    row_count[i] += 1
                    col_count[j] += 1

    #satisfy min regulations for gene
    for j in range(n):
        while col_count[j] < min_out:
            i = random.randrange(n)
            if (i,j) not in chosen and i != forced and i != j:
                chosen.add((i, j))
                row_count[i] += 1
                col_count[j] += 1

    #if there are left over edges, distribute
    eligible = [(i, j) for i in range(n) if i != forced for j in range(n) if (i, j) not in chosen and i != j]
    left = edges - len(chosen)
    if left > 0:
        for (i, j) in random.sample(eligible, left):
            chosen.add((i, j))
    for (i, j) in chosen:
        M[i, j] = get_lam(0.5)
    return M

#build pair of similar networks
#enforce: overlapping edges must have the same sign (activation vs inhibition)
def build_pair(n, edges, similarity = 0.7, min_in = 1, min_out = 1,forced = 3):
    M = build_edges(n, edges, forced, min_in, min_out)
    #number of edges to be deleted then replaced with new, distinct edges
    r = int(round(edges * (1-similarity)/(1+similarity)))
    in_one = set((i,j) for i in range(n) for j in range(n) if M[i, j] != 1)
    eligible = set((i,j) for i in range(n) for j in range(n) if (i,j) not in in_one and i != forced and i!=j)

    # original regulators per gene (row i)
    ob = [set() for _ in range(n)]
    for i, j in in_one:
        ob[i].add(j)
        
    while True:
        #randomly delete + re-add r genes until all requirements are satisfied
        del_set = set(random.sample(list(in_one), r))
        new_set = set(random.sample(list(eligible), r))
        # overlap constraint: don't delete all regulators for any gene i (i != forced)
        edges_new = (in_one - del_set) | new_set
    
        #check if ob_new follows requirements
        row_ct = [0]*n
        col_ct = [0]*n
        nb = [set() for _ in range(n)]
        for i, j in edges_new:
            row_ct[i] += 1
            col_ct[j] += 1
            nb[i].add(j)
        if any(i != forced and not (ob[i] & nb[i]) for i in range(n)):
            continue

        if all(row_ct[i] >= min_in for i in range(n) if i != forced) and all(col_ct[j] >= min_out for j in range(n)):
            M2 = np.ones((n, n), dtype=float)
            overlap = edges_new & in_one
            for i, j in overlap:
                if M[i, j] < 1:
                    M2[i, j] = log_rand(0.05, 0.5)
                else:
                    M2[i, j] = log_rand(2, 20)
            for i, j in (edges_new - in_one):
                M2[i, j] = get_lam(0.5)

            return M, M2

## Simulating a pair

In [ ]:
#wrapper combining randomly generated lambda values + update_params
def simulate_random(n, lam, forced = 3, forced_value = 25, time = 1000, steps = 1000):
    #clean lambda values
    lam[forced, :] = 1
    np.fill_diagonal(lam, 1)
    
    g = np.array([log_rand(2, 8) for i in range(n)])
    k = np.array([log_rand(0.1, 0.5) for i in range(n)])
    R = np.array([[log_rand(1, 100) for i in range(n)] for j in range(n)])
    n_par = np.array([[log_rand(1, 5) for i in range(n)] for j in range(n)])
    params = {"g": g, "k": k, "l": lam, "R": R, "n": n_par}

    x_init, params, df = update_params(params, forced, forced_value, time, steps)
    return x_init, params, df

#get 2 random pairs + simulate/plot both
def simulate_pair(n, edges, similarity = 0.7, min_in = 1, min_out = 1, 
                  forced = 3, forced_value = 25, time = 1000, steps = 1000, seed = 123):
    random.seed(seed)
    lam1, lam2 = build_pair(n, edges, similarity, min_in, min_out, forced)
    x_init1, param1, df1 = simulate_random(n, lam1, forced, forced_value, time, steps)
    x_init2, param2, df2 = simulate_random(n, lam2, forced, forced_value, time, steps)
    return x_init1, x_init2, param1, param2, df1, df2

## Screening metrics

In [ ]:
def compute_metrics(df):
    x = df.to_numpy(float)

    # scale each gene to 0–1
    mn = x.min(axis=1, keepdims=True)
    mx = x.max(axis=1, keepdims=True)
    x_scaled = (x - mn) / (mx - mn)

    # turning points via confirmed pivots (endpoints excluded by construction)
    delta = 0.01
    tp = 0

    for gene in range(x_scaled.shape[0]):
        y = x_scaled[gene]
        direction = 0
        ext_val = y[0]
        for k in range(1, y.size):
            if direction == 0:
                # establish direction only after moving >= delta from the start value
                if y[k] - y[0] >= delta:
                    direction = 1
                    ext_val = y[k]
                elif y[0] - y[k] >= delta:
                    direction = -1
                    ext_val = y[k]
            elif direction == 1:  # trending up: track max, confirm peak after drop >= delta
                if y[k] >= ext_val:
                    ext_val = y[k]
                elif ext_val - y[k] >= delta:
                    direction = -1
                    ext_val = y[k]
                    tp += 1
            else:  # direction == -1 trending down: track min, confirm trough after rise >= delta
                if y[k] <= ext_val:
                    ext_val = y[k]
                elif y[k] - ext_val >= delta:
                    direction = 1
                    ext_val = y[k]
                    tp += 1
    tp /= x_scaled.shape[0]
    # if avg tp > 2, there are likely oscillations/not smooth trajectories
    if tp > 4:
        tp = 0

    # nearest-neighbor cosine similarity
    xc = x_scaled - x_scaled.mean(axis=1, keepdims=True)
    best = 0.0
    for i in range(xc.shape[0]):
        for j in range(xc.shape[0]):
            if i == j:
                continue
            similarity = abs((xc[i] @ xc[j]) /
                             (np.linalg.norm(xc[i]) * np.linalg.norm(xc[j])))
            if similarity > best:
                best = similarity
    
    # max slope control (scaled units per time-step)
    max_slope = 0.0
    for gene in range(x_scaled.shape[0]):
        y = x_scaled[gene]
        for k in range(1, y.size):
            s = abs(y[k] - y[k-1])
            if s > max_slope:
                max_slope = s
                
    return tp, float(best), float(max_slope)

## Ground truth, decoy padding and output scaling

In [ ]:
#in entry [i, j], i is the target, j is the source
def make_ground_truth(n, lam):
    rows = []
    for i in range(n): 
        for j in range(n):
            if lam[i, j] != 1.0:
                if lam[i, j] > 1.0:
                    t = 1
                else:
                    t = 2
                rows.append([f"gene_{j+1}", f"gene_{i+1}", t])
    return pd.DataFrame(rows, columns=["Source", "Target", "Interaction"])


def make_initial_network(n, lam, k, seed):
    random.seed(seed)

    # --- ground-truth edges (i=target, j=source) ---
    gt_edges = set()
    indeg = [0] * n
    for i in range(n):
        for j in range(n):
            if lam[i, j] != 1.0:
                gt_edges.add((i, j))
                indeg[i] += 1

    # --- reserve some false edges to fix zero-indegree targets ---
    missing_targets = [i for i in range(n) if indeg[i] == 0]
    extra_fixed = set()
    for i in missing_targets:
        # pick a source j != i that isn't already an edge
        candidates = [j for j in range(n) if j != i and (i, j) not in gt_edges and (i, j) not in extra_fixed]
        j = random.choice(candidates)
        extra_fixed.add((i, j))
        indeg[i] += 1

    # --- sample remaining false edges uniformly ---
    remaining_k = k - len(extra_fixed)
    eligible = [(i, j) for i in range(n) for j in range(n)
                if i != j and (i, j) not in gt_edges and (i, j) not in extra_fixed]
    extra_random = set(random.sample(eligible, remaining_k))
    all_edges = list(gt_edges | extra_fixed | extra_random)
    all_edges.sort()
    rows = [[f"gene_{j+1}", f"gene_{i+1}", 1] for (i, j) in all_edges]
    return pd.DataFrame(rows, columns=["Source", "Target", "Interaction"])

def process_raw_df(raw_df):
    x = raw_df.to_numpy()
    mn = x.min(axis=1, keepdims=True)
    mx = x.max(axis=1, keepdims=True)
    x = 50.0 * (x - mn) / (mx - mn) 
    x = np.log2(x + 1.0)
    return pd.DataFrame(x, index=raw_df.index, columns=raw_df.columns)

## The full generation workflow

In [ ]:
def run_full_workflow(n, edges, num_models, fake_edges_list):
    num_seeds = 2000
    similarity = 0.7
    min_in = 1
    min_out = 1
    forced = 3
    forced_value = 25
    time = 1000
    steps = 1000
    #bottom 1% similarity, bottom 80% max slope
    #(drop TP requirement)
    top_tp = 0.2
    bottom_sim = 0.01
    bottom_slope = 0.8
    # ----------------------------------------------------------
    # find cutoffs
    tps = np.empty(num_seeds)
    avg_simils = np.empty(num_seeds)
    max_slopes = np.empty(num_seeds)
    for i in range(num_seeds):
        x_init1, x_init2, param1, param2, df1, df2 = simulate_pair(n, edges, seed = i)
        tp1, avg_sim1, max_slope1 = compute_metrics(df1)
        tp2, avg_sim2, max_slope2 = compute_metrics(df2)
        tps[i] = min(tp1, tp2)
        avg_simils[i] = max(avg_sim1, avg_sim2)
        max_slopes[i] = max(max_slope1, max_slope2)
        #if i % 500 == 0:
        #    print(i/500)
    k_tp = int(np.ceil(top_tp * num_seeds))
    k_sim = int(np.ceil(bottom_sim * num_seeds))
    k_slp = int(np.ceil(bottom_slope * num_seeds))
    tp_cutoff    = np.partition(tps, -k_tp)[-k_tp]   
    sim_cutoff   = np.partition(avg_simils, k_sim-1)[k_sim-1] 
    slope_cutoff = np.partition(max_slopes, k_slp-1)[k_slp-1] 
    print(tp_cutoff)
    print(sim_cutoff)
    print(slope_cutoff)

    # find good seeds
    good_seeds = []
    seed = 0
    while len(good_seeds) < num_models:
        x_init1, x_init2, param1, param2, df1, df2 = simulate_pair(n, edges, seed = seed)
        tp1, avg_sim1, max_slope1 = compute_metrics(df1)
        tp2, avg_sim2, max_slope2 = compute_metrics(df2)
        if max(avg_sim1, avg_sim2) < sim_cutoff and max(max_slope1, max_slope2) < slope_cutoff:
            good_seeds.append(seed)
            print(seed)
        seed += 1   

    # export data
    base_dir = Path(".")
    fake_edges_list = sorted({int(k) for k in fake_edges_list})

    for seed in good_seeds:
        case_dir = base_dir / f"ODE_Case_{seed}"
        case_dir.mkdir(parents=True, exist_ok=True)
        # simulate both networks for this seed
        x_init1, x_init2, param1, param2, df1, df2 = simulate_pair(n, edges,
            similarity=similarity, min_in=min_in, min_out=min_out,
            forced=forced, forced_value=forced_value,
            time=time, steps=steps,
            seed=seed)

        lam1 = np.asarray(param1["l"])
        lam2 = np.asarray(param2["l"])

        # ground truths
        gt1 = make_ground_truth(n, lam1)
        gt2 = make_ground_truth(n, lam2)

        # processed dfs
        proc1 = process_raw_df(df1)
        proc2 = process_raw_df(df2)

        # write files
        gt1.to_csv(case_dir / "ground_truth_1.csv", index=False)
        gt2.to_csv(case_dir / "ground_truth_2.csv", index=False)
        proc1.to_csv(case_dir / "data_1.csv", index=True)
        proc2.to_csv(case_dir / "data_2.csv", index=True)
        
        for k_false in fake_edges_list:
            total_edges = edges + k_false
            init_dir = case_dir/f"total_{total_edges}_edges"
            init_dir.mkdir(parents=True, exist_ok=True)
            
            init1 = make_initial_network(n, lam1, k_false, seed=seed)
            init2 = make_initial_network(n, lam2, k_false, seed=seed + 1)
            
            init1.to_csv(init_dir / "initial_network_1.csv", index=False)
            init2.to_csv(init_dir / "initial_network_2.csv", index=False)
    return good_seeds, tp_cutoff, sim_cutoff, slope_cutoff

## Run

In [ ]:
n = 7
edges = 18
num_models = 20
fake_edges_list = [9, 12, 15, 18, 21, 24]
run_full_workflow(n, edges, num_models, fake_edges_list)